## Ultralytics YOLO를 사용한 다중 객체 추적
### Ultralytics YOLO 기반 트랙킹(추적)은 영상에서 객체를 탐지 + ID를 붙여 물체를 계속 추적하는 기술
#### 2단계로 작업
1. Detection : 사람 / 동물 / 자동차 등 감지
2. Tracking : 같은 객체에 ID를 부여해 프레임이 바뀌어도 동일 객체 유지

#### 내부 알고리즘
- Bot-sort
- ByteTrack <- 우리는 이걸 사용

#### 장점
- ID 유지: 동일 객체에 일관된 ID
- 예측 가능: 다음 위치 예측
- 안정성: 일시적 가림 대응
- 궤적 분석: 이동 경로 파악

In [1]:
!pip install ultralytics opencv-python

In [2]:
import cv2
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

video_path = 'road_car.mp4'

cap = cv2.VideoCapture(video_path) # 지정한 동영상 파일을 프레임 단위로 읽어 처리함.

if not cap.isOpened():
  print('동영상 읽기 실패')
  exit()

while True:
  ret, frame = cap.read() # ret:읽기 성공 여부, frame:실제 이미지 데이터

  if not ret:
    break # 프레임을 읽지 못하면 빠져나옴.

  results = model.track(
      source = frame, # 분석할 입력 이미지 데이터 -> 현재 읽은 1장의 프레임 이미지
      persist= True,  # 추정 정보 유지
      # 객체 추적시 사용할 알고리즘 선택(botsort.yaml, bytetrack.yaml)
      # 현재 프레임과 이전 프레임을 비교해 객체에 ID를 부여함
      tracker = "bytetrack.yaml",
      verbose = False,
      # show = True     # 탐지 추적결과를 화면으로 보여줌(객체박스 + id)
  )

  result = results[0] # YOLO결과 리스트에서 첫번째거 하나만 꺼냄

  # 결과값에 box와 id가 있는 경우 진행
  if result.boxes is not None and result.boxes.id is not None:
    boxes = result.boxes.xyxy.cpu().numpy()
    # 추적 ID -> 같은 차량은 Frame이 바뀌어도 같은 ID
    ids = result.boxes.id.cpu().numpy().astype(int)
    # Class ID 얻기 ex) car = 2
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    # Box좌표, 추적 ID, Class ID를 묶어 반복처리하기
    for box, track_id, class_id in zip(boxes, ids, class_ids):
      x1, y1, x2, y2 = map(int, box) # 실수 좌표를 정수 좌표로 얻음.
      # class 번호에 해당하는 class name얻기
      class_name = result.names[class_id] # ex) 2 => car
      print(f'id:{track_id}, class:{class_name}, Box(x1:{x1}, y1:{y1}, x2:{x2}, y2:{y2})')

  annotated_frame = result.plot() # 욜로가 탐지한 이미지 자동으로 그리기 - 각종 정보가 표시

  # 크기 줄이기
  display_frame = cv2.resize(annotated_frame, (960, 540))

  cv2.imshow('YOLO Tracking', display_frame)

  if cv2.waitKey(1) & 0xFF == ord('q'):
    break

cap.release()
cv2.destroyAllWindows() # OpenCV로 연창 모두

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 43.8 MB/s  0:00:00

requirements: AutoUpdate success  2.0s
WARNING requirements: Restart runtime or rerun command for updates to take effect

id:1, class:car, Box(x1:1155, y1:1309, x2:1646, y2:1791)
id:2, class:car, Box(x1:761, y1:1373, x2:1435, y2:1975)
id:3, class:car, Box(x1:506, y1:1962, x2:1172, y2:2155)
id:4, class:car, Box(x1:2204, y1:1167, x2:2776, y2:1740)
id:5, class:car, Box(x1:2039, y1:877, x2:2432, y2:1219)
id:6, class:car, Box(x1:1252, y1:1014, x2:1743, y2:1433)
id:7, class:car, Box(x1:0, y1:689, x2:359, y2:981)
id:8, class:car, Box(x1:1295, y1:835, x2:1660, y2:1054)
id:9, class:truck, Box(x1:2248, y1:1697, x2:3313, y2:2153)
id:10, class:truck, Box(x1:2125, y1:448, x2:2416, y2:699)
id:11, class:car, Box(x1:1483, y1:764, x2:1790, y2:1058)
id:12, class:car, Box(x1